# Non-stationary Markov Chain Attribution for Heatwaves over Europe

This notebook runs the full attribution analysis pipeline:

1. **Configuration** – set all paths and model parameters
2. **Model estimation** – fit non-stationary first-order Markov-GPD models (Model 1 & 2)
3. **Attribution** – compute likelihood ratios for the ERA5 time series

All input data must be available in `epi_output/`. No additional download required.

**Assumptions:** first-order Markov chain, no seasonal cycle, no ensemble bootstrapping.

**Reference:** Meurer et al. (2025) – *Non-stationary time series attribution for heatwaves over Europe*

## 1. Imports

In [1]:
import concurrent.futures
import glob
import os
import time
import warnings

import matplotlib.pyplot as plt
from tqdm import tqdm

from markov_functions import attribution_analysis

warnings.filterwarnings("ignore")

## 2. Configuration

Adjust paths and parameters here before running the analysis.

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DIR  = "/home/b/b382884/Non-stationary-time-series-attribution-for-heatwaves-over-Europe/epi_output/"
OUTPUT_DIR = "/work/bm1159/XCES/xces-work/b382884/MYWORK/git_code_test"

# ── Analysis settings ─────────────────────────────────────────────────────────
REGION           = "reg_19"   # one of: reg_16, reg_17, reg_19
THRESHOLD        = 0.95       # quantile threshold for extreme value definition
MAX_DEGREE       = 5          # max Legendre polynomial degree (GPD)
MAX_DEGREE_ALPHA = 5          # max Legendre polynomial degree (copula)
FOLDERS_HIST     = True       # data stored in subdirectories (hist)
FOLDERS_HISTNAT  = True       # data stored in subdirectories (histnat)

# ── ERA5: auto-detect .nc file inside ERA5/<system-id>/ ───────────────────────
era5_files = glob.glob(os.path.join(INPUT_DIR, REGION, "ERA5", "*", "*.nc"))
if not era5_files:
    raise FileNotFoundError(
        f"No ERA5 .nc file found under {os.path.join(INPUT_DIR, REGION, 'ERA5')}"
    )
ERA5_PATH = era5_files[0]
print(f"ERA5 file : {ERA5_PATH}")

# ── CMIP6 models and number of SSP years appended to hist run ─────────────────
MODELS = ["access", "bcc", "canesm5", "cnrm", "hadgem3", "ipsl", "miroc6", "mpi", "mri"]

SSP_YEARS = {
    "hadgem3" : 8,
    "bcc"     : 10,
    "canesm5" : 10,
    "miroc6"  : 10,
    "access"  : 11,
    "cnrm"    : 11,
    "ipsl"    : 11,
    "mpi"     : 11,
    "mri"     : 11,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

ERA5 file : /home/b/b382884/Non-stationary-time-series-attribution-for-heatwaves-over-Europe/epi_output/reg_19/ERA5/943352/ExtremalPatternIndex_TPDM.nc
Output directory: /work/bm1159/XCES/xces-work/b382884/MYWORK/git_code_test


## 3. Analysis function

Wraps both model steps for a single CMIP6 model.

In [3]:
def run_model_analysis(model_name: str) -> None:
    """Run the full attribution analysis (Model 1 + Model 2) for one CMIP6 model.

    Parameters
    ----------
    model_name:
        Name of the CMIP6 model (must be a key in SSP_YEARS).
    """
    folder_hist    = os.path.join(INPUT_DIR, REGION, f"{model_name}_hist")
    folder_histnat = os.path.join(INPUT_DIR, REGION, f"{model_name}_histnat")

    for folder in [folder_hist, folder_histnat]:
        if not os.path.exists(folder):
            raise FileNotFoundError(f"Input folder not found: {folder}")

    output_dir = os.path.join(OUTPUT_DIR, REGION, "without_seasonal_cycle", model_name)
    os.makedirs(output_dir, exist_ok=True)

    analysis = attribution_analysis(
        main_folder_hist    = folder_hist,
        main_folder_histnat = folder_histnat,
        q                   = THRESHOLD,
        model_name          = model_name,
        max_degree          = MAX_DEGREE,
        max_degree_alpha    = MAX_DEGREE_ALPHA,
        folders_hist        = FOLDERS_HIST,
        folders_histnat     = FOLDERS_HISTNAT,
        output_dir          = output_dir,
        num_of_years_ssp    = SSP_YEARS[model_name],
        era5_file_path      = ERA5_PATH,
    )

    print(f"[{model_name}] Starting – region: {REGION}")
    t0 = time.time()
    analysis.run_analysis_model1()
    analysis.run_analysis_model2()
    print(f"[{model_name}] Done in {time.time() - t0:.1f} s")

## 4. Run analysis (parallel over all models)

In [ ]:
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = {executor.submit(run_model_analysis, model): model for model in MODELS}
    for future in tqdm(concurrent.futures.as_completed(futures),
                       total=len(MODELS), desc="Models"):
        model = futures[future]
        try:
            future.result()
        except Exception as exc:
            print(f"[{model}] FAILED: {exc}")

Models:   0%|          | 0/9 [00:00<?, ?it/s]

[bcc] Starting – region: reg_19[access] Starting – region: reg_19[canesm5] Starting – region: reg_19[hadgem3] Starting – region: reg_19[cnrm] Starting – region: reg_19[ipsl] Starting – region: reg_19[mpi] Starting – region: reg_19[miroc6] Starting – region: reg_19[mri] Starting – region: reg_19







[bcc] Starting Model 1 analysis...


[access] Starting Model 1 analysis...
[canesm5] Starting Model 1 analysis...
[cnrm] Starting Model 1 analysis...
[hadgem3] Starting Model 1 analysis...
[mpi] Starting Model 1 analysis...
[miroc6] Starting Model 1 analysis...
[ipsl] Starting Model 1 analysis...


[mri] Starting Model 1 analysis...






Optimization terminated successfully.Optimization terminated successfully.

         Current function value: 0.198492
         Current function value: 0.198512
         Iterations 7         Iterations 7

Optimization terminated successfully.Optimization terminated successfully.
Optimization terminated successfully.
         Current function value: 0.1981

## 5. Check output

In [ ]:
result_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.nc"), recursive=True))
print(f"Found {len(result_files)} NetCDF output files:")
for f in result_files:
    print(" ", f)

## 6. Software versions

In [ ]:
import sys

used_modules = ["cartopy", "matplotlib", "numpy", "pandas",
                "scipy", "statsmodels", "tqdm", "xarray"]

print(f"Python {sys.version}\n")
for name in used_modules:
    mod = sys.modules.get(name)
    version = getattr(mod, "__version__", "n/a") if mod else "not loaded"
    print(f"  {name:<20} {version}")